# Stage 4: semantic segmentation of breast tissue

This is the one stage of the pipeline that cannot run on the analysis machine.
It needs a GPU with several GB of memory, and it needs annotated tissue.

**Run this on Colab with a GPU runtime:** Runtime, Change runtime type, T4 GPU.

## What this does, and why each choice was made

The published method trains DeepLab v3+ to label ten tissue classes, using
roughly 50 hand-drawn examples of each class per image. No such annotation
exists for the slides in this study and it cannot be manufactured, so the
training data comes instead from **BCSS**, which provides over 20,000
pixel-level tissue annotations across 151 TCGA-BRCA slides. Those are the same
slides the analysis cohort is drawn from, so the model is validated on the
tissue it is applied to rather than on a different set of patients.

Amgad M et al. Structured crowdsourcing enables convolutional segmentation of
histology images. *Bioinformatics* 2019;35:3461-3467.

## The trap in this dataset, stated by its authors and honoured here

Pixel value 0 is **outside the region of interest**, and value 7 is
**exclude**. Neither is an "other tissue" class. Training on them as if they
were a class teaches the model that unannotated background is a tissue type,
which then appears confidently all over the whole-slide inference. They are
given zero weight in the loss, and every metric below is computed only over
annotated pixels.

In [ ]:
# --- 1. environment -------------------------------------------------------
import subprocess, sys, torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')
print(torch.cuda.get_device_name(0),
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
subprocess.run([sys.executable,'-m','pip','install','-q','girder-client','segmentation-models-pytorch','albumentations'])

In [ ]:
# --- 2. fetch the prepared BCSS tiles ------------------------------------
# The girder folder referenced by the BCSS download script holds WHOLE SLIDE
# IMAGES, not tiles: that script downloads each WSI, pulls the annotation
# polygons, rasterises them and crops matching RGBs. That is many gigabytes and
# unnecessary here, because the authors also publish the finished dataset at
# 0.25 micron per pixel on Google Drive. That copy is used.
#
# An earlier version of this cell looked for 'rgbs' and 'masks' subfolders that
# do not exist, downloaded nothing, and reported success. The failure only
# appeared two cells later as an empty train/test split, far from its cause, so
# this cell now verifies that pairs actually exist before letting the run go on.
import subprocess, sys, os
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
import gdown

OUT = Path('/content/bcss'); OUT.mkdir(parents=True, exist_ok=True)
FOLDER_ID = '1zqbdkQF8i5cEmZOGmbdQm-EP8dRYtvss'   # BCSS at 0.25 MPP, public

if not any(OUT.rglob('*.png')):
    gdown.download_folder(id=FOLDER_ID, output=str(OUT), quiet=False,
                          use_cookies=False, remaining_ok=True)

pngs = list(OUT.rglob('*.png'))
print(f'{len(pngs)} png files downloaded')
dirs = sorted({p.parent.name for p in pngs})
print('folders:', dirs)

# locate the image and mask directories whatever they are called
def pick(*words):
    for d in dirs:
        if any(w in d.lower() for w in words):
            return d
    return None
RGB_DIR, MASK_DIR = pick('rgb', 'image'), pick('mask', 'label')
print('rgb dir :', RGB_DIR, '
mask dir:', MASK_DIR)

assert pngs, ('No files downloaded. Open the Drive folder in a browser to check '
              'it is still shared: '
              'https://drive.google.com/drive/folders/' + FOLDER_ID)
assert RGB_DIR and MASK_DIR, (
    f'Could not identify image and mask folders among {dirs}. '
    'Set RGB_DIR and MASK_DIR by hand from that list and re-run this cell.')

In [ ]:
# --- 3. tiles, with the class grouping the BCSS baseline specifies --------
# 22 raw classes are heavily imbalanced and several are rare enough that a
# per-class metric on them would be noise. The published baseline groups them,
# and the same grouping is used here so results are comparable to it.
import numpy as np, cv2, re
from pathlib import Path

GROUPS = {1:1, 20:1,                     # tumor, dcis      -> tumour
          2:2, 12:2,                     # stroma, mucoid   -> stroma
          3:3, 10:3, 11:3,               # lymphocytic, plasma, other immune
          4:4,                           # necrosis or debris
          9:5, 13:5, 16:5, 17:5, 18:5, 14:5, 6:5, 5:5, 8:5, 19:5, 21:5, 15:5}
NAMES = {1:'tumour', 2:'stroma', 3:'inflammatory', 4:'necrosis', 5:'other'}
IGNORE = 255                              # 0 outside_roi and 7 exclude map here
TILE, STRIDE = 512, 384

def regroup(m):
    out = np.full(m.shape, IGNORE, np.uint8)
    for src, dst in GROUPS.items():
        out[m == src] = dst
    return out                            # 0 and 7 never assigned -> stay IGNORE

pairs = []
rgb_files = [p for p in Path('/content/bcss').rglob('*.png')
             if p.parent.name == RGB_DIR]
mask_by_name = {p.name: p for p in Path('/content/bcss').rglob('*.png')
                if p.parent.name == MASK_DIR}
for r in sorted(rgb_files):
    m = mask_by_name.get(r.name)
    if m is not None: pairs.append((r, m))
print(len(pairs), 'image/mask pairs')

# tile once, recording the slide each tile came from so the split can respect it
X, Y, C = [], [], []
for r, m in pairs:
    mo = re.match(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', r.name)
    case = mo.group(1) if mo else 'NA'
    img = cv2.cvtColor(cv2.imread(str(r)), cv2.COLOR_BGR2RGB)
    msk = regroup(cv2.imread(str(m), cv2.IMREAD_GRAYSCALE))
    H, W = msk.shape
    for y in range(0, max(H-TILE,0)+1, STRIDE):
        for x in range(0, max(W-TILE,0)+1, STRIDE):
            t = msk[y:y+TILE, x:x+TILE]
            if t.shape != (TILE,TILE): continue
            if (t != IGNORE).mean() < 0.35:   # mostly unannotated, skip
                continue
            X.append(img[y:y+TILE, x:x+TILE]); Y.append(t); C.append(case)
X, Y, C = np.array(X), np.array(Y), np.array(C)
print('tiles', X.shape, 'from', len(np.unique(C)), 'slides')
vals, cnt = np.unique(Y[Y!=IGNORE], return_counts=True)
for v,c in zip(vals,cnt): print(f'  {NAMES.get(v,v):14s} {100*c/cnt.sum():5.1f} %')
assert pairs, ('No image/mask pairs matched by filename. RGB and mask files must share a name; inspect a few names in each folder and adjust.')

In [ ]:
# --- 4. split by SLIDE, never by tile ------------------------------------
# Tiles from one slide share staining, scanner and patient. Splitting at tile
# level puts neighbouring tiles in both train and test and reports a score that
# measures memorisation rather than generalisation. The split is by slide, and
# the assertion below fails the run rather than letting a leak pass silently.
uc = np.unique(C)
rng = np.random.default_rng(0); rng.shuffle(uc)
n_test = max(1, len(uc)//5)
test_cases = set(uc[:n_test])
te = np.isin(C, list(test_cases)); tr = ~te
print(f'{len(uc)} slides -> train {tr.sum()} tiles / {len(uc)-n_test} slides, '
      f'test {te.sum()} tiles / {n_test} slides')
assert not (set(C[tr]) & set(C[te])), 'a slide appears in both splits'
assert te.sum() > 0 and tr.sum() > 0, 'empty split'

In [ ]:
# --- 5. train DeepLab v3+ -------------------------------------------------
import torch, torch.nn as nn, segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485,0.456,0.406]); STD = np.array([0.229,0.224,0.225])
class DS(Dataset):
    def __init__(s, X, Y, aug): s.X, s.Y, s.aug = X, Y, aug
    def __len__(s): return len(s.X)
    def __getitem__(s, i):
        x, y = s.X[i], s.Y[i]
        if s.aug:
            if np.random.rand() < .5: x, y = x[:, ::-1], y[:, ::-1]
            if np.random.rand() < .5: x, y = x[::-1], y[::-1]
            k = np.random.randint(4); x, y = np.rot90(x,k), np.rot90(y,k)
        x = ((x/255.0 - MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy().astype(np.int64))

model = smp.DeepLabV3Plus('resnet34', encoder_weights='imagenet',
                          classes=6).cuda()   # index 0 unused; classes 1..5
# ignore_index is what keeps 'outside roi' and 'exclude' out of the loss
crit = nn.CrossEntropyLoss(ignore_index=IGNORE)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
scaler = torch.cuda.amp.GradScaler()
dl_tr = DataLoader(DS(X[tr],Y[tr],True), batch_size=8, shuffle=True, num_workers=2, drop_last=True)
dl_te = DataLoader(DS(X[te],Y[te],False), batch_size=8, num_workers=2)

EPOCHS = 12
for ep in range(EPOCHS):
    model.train(); tot = 0
    for xb, yb in dl_tr:
        xb, yb = xb.cuda(), yb.cuda()
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            loss = crit(model(xb), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item()
    print(f'epoch {ep:2d}  loss {tot/len(dl_tr):.4f}')

In [ ]:
# --- 6. validate: confusion matrix and the 90 percent acceptance gate -----
# The published protocol accepts a model only when every class reaches 90
# percent precision and recall, and adds annotations and retrains otherwise.
# Classes that fail are named rather than averaged away.
import pandas as pd
model.eval(); cm = np.zeros((6,6), np.int64)
with torch.no_grad():
    for xb, yb in dl_te:
        p = model(xb.cuda()).argmax(1).cpu().numpy()
        y = yb.numpy(); keep = y != IGNORE
        np.add.at(cm, (y[keep], p[keep]), 1)
rows = []
for c in range(1,6):
    tp = cm[c,c]; fp = cm[:,c].sum()-tp; fn = cm[c,:].sum()-tp
    prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1)
    rows.append({'class':NAMES[c],'support':int(cm[c,:].sum()),
                 'precision':round(100*prec,1),'recall':round(100*rec,1),
                 'passes_90_gate': bool(prec>=.9 and rec>=.9)})
df = pd.DataFrame(rows); print(df.to_string(index=False))
acc = np.trace(cm)/max(cm.sum(),1)
print(f'\noverall accuracy on annotated pixels: {100*acc:.1f} %')
failed = df[~df.passes_90_gate]['class'].tolist()
print('classes FAILING the 90 percent gate:', failed if failed else 'none')
df.to_csv('/content/stage4_per_class_metrics.csv', index=False)
pd.DataFrame(cm[1:,1:], index=[NAMES[i] for i in range(1,6)],
             columns=[NAMES[i] for i in range(1,6)]).to_csv('/content/stage4_confusion.csv')

In [ ]:
# --- 7. export for CPU inference on the analysis machine ------------------
torch.save({'state_dict': model.state_dict(), 'arch':'DeepLabV3Plus',
            'encoder':'resnet34', 'classes':6, 'names':NAMES,
            'ignore_index':IGNORE, 'tile':TILE,
            'test_cases':sorted(test_cases)}, '/content/stage4_deeplab.pt')
from google.colab import files
for f in ['/content/stage4_deeplab.pt','/content/stage4_per_class_metrics.csv',
          '/content/stage4_confusion.csv']:
    files.download(f)
print('Put stage4_deeplab.pt in  data/models/  on the analysis machine.')
print('Inference then runs on CPU: the model is ~90 MB and a slide is tiled lazily.')